## **Tikhonov Regularisation**

In this notebook, we will see if Tikhonov regularisation worsens or improves the parameter recovery.

One inversion, at a single $\lambda = 10^{-6}$, started from the same point as run 1
of the ensemble in `averaged-runs.ipynb` -- the run that reached the lowest data
misfit there. The data, the grid, the sources, the noise and the solver settings are
that notebook's, so the penalty is the only thing that differs and the recovered
parameters can be read against its table directly.

In [1]:
# the whole API the experiment needs

import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch

import neuralphysics
from neuralphysics import (
    LIMB,
    SOURCE_SPECS,
    CoefficientHierarchy,
    ElectrodeArray,
    Observation,
    add_measurement_noise,
    configure,
    dipole_centres,
    forward,
    gaussian_dipole,
    grid_at,
    invert,
    laplacian_metric_3D,
    levels,
    parameter_names,
    sample_initial_parameters,
    thicknesses,
    trapezoid_weight_field,
    true_sigma,
    unpack,
    viz,
)

In [2]:
print(configure())
print(f"neuralphysics {neuralphysics.__version__} | torch {torch.__version__} | "
      f"numpy {np.__version__} | scipy {scipy.__version__} | python {platform.python_version()}")
print(f"default dtype {torch.get_default_dtype()} | torch threads {torch.get_num_threads()} | "
      f"{platform.platform()}")

{'dtype': 'torch.float64', 'seed': 42}
neuralphysics 0.1.0 | torch 2.5.1 | numpy 1.26.4 | scipy 1.17.1 | python 3.11.15
default dtype torch.float64 | torch threads 4 | macOS-26.6.1-arm64-arm-64bit


In [ ]:
# the forward experiment
# anything not named here is at the package default
# L = 1, z_zeta = 1, coarsening floor of 9, jacobi damping factor omega = 2/3, nu = (2, 2)
# 50 base iterations, and 100% electrode coverage on the four lateral faces

N = 65 # nodes per axis; N = 2^k + 1 must hold for the v-cycle to work
NUM_CYCLES = 8 # fixed budget - not a convergence tolerance
NOISE_PERCENT = 5.0 # noise std as a percentage of the clean field's volume RMS
NOISE_SEED = 0

In [4]:
# the inverse problem

# The starting point is not chosen here. It is run 1 of the ensemble in
# averaged-runs.ipynb, the run that reached the lowest data misfit there, and it is
# reproduced from that notebook's seed rather than copied across as numbers: a
# transcribed start would silently stop matching the moment the draw changed, and
# the whole point of starting here is that this run and that one begin in the same
# place. The four constants below are that notebook's, and have to stay its.
ENSEMBLE_SEED = 12345      # seeds the draws, and nothing else
N_RUNS = 5                 # how many were drawn there. The draws are sequential, so
                           # this must match for run 1 to be the same vector
LOG_SPREAD = np.log(3.0)
RADIUS_SPREAD_MM = 1.5

RUN = 1                    # which of those draws to start from

# the one thing this notebook changes about that run: a penalty on the
# log-conductivities wandering from where they started,
# J_lambda(theta) = J(theta) + (lambda/2) ||rho - rho_ref||^2, with rho_ref the
# starting rho. The radii are not penalised -- they are locations rather than
# scales, and a geometry prior is a different claim from a conductivity one.
TIKHONOV = 1e-6

# the global L-BFGS-B iteration budget, shared across the restarts of one run.
# 200 because that is averaged-runs.ipynb's: a different budget would be a second
# difference between the two runs, and the comparison this notebook makes needs
# the penalty to be the only one.
MAX_ITERS = 200

# scipy's tolerances are calibrated for O(1) objectives. Here J ~ 1e-5 and
# ||grad J|| ~ 5e-6 near the solution, so the default gtol = 1e-5 is already satisfied
# at the starting point and L-BFGS-B would return after one evaluation having done
# nothing. ftol is relative to max(|f|, 1) = 1, so it is effectively absolute, and the
# default 2.2e-9 would stop while the misfit is still moving in the fourth digit.
FTOL = 1e-16
GTOL = 1e-12

LIMB_RADIUS_MM = 24.0   # the physical limb radius, at r/R = 0.5
MM_PER_UNIT = LIMB_RADIUS_MM / 0.5   # normalised radius -> millimetres

RESULTS = Path("..") / "experiment-results"   # where the report picks the table up

In [5]:
print(f"grid     N = {N}, levels {levels(N)}")
print(f"solver   {NUM_CYCLES} V-cycles")
print(f"data     {NOISE_PERCENT}% noise (seed {NOISE_SEED})")
print(f"inverse  L-BFGS-B restarted to a fixed point, <= {MAX_ITERS} iterations, "
      f"ftol {FTOL:g}, gtol {GTOL:g}")
print(f"penalty  Tikhonov lambda = {TIKHONOV:g} on the log-conductivities")

grid     N = 65, levels [65, 33, 17, 9]
solver   8 V-cycles
data     5.0% noise (seed 0)
inverse  L-BFGS-B restarted to a fixed point, <= 200 iterations, ftol 1e-16, gtol 1e-12
penalty  Tikhonov lambda = 1e-06 on the log-conductivities


In [6]:
# the tissue table - read off the model
print(f"{'layer':8s} {'r/R':>13s} {'thickness':>12s} {'sigma_T':>9s} {'sigma_L':>9s}   source")
r_lo = 0.0
for nm, r_hi in zip(LIMB.layer_names, LIMB.interfaces + (0.5,)):
    sT, sL = LIMB.sigma_layers[nm]
    print(f"{nm:8s} {f'{r_lo:.3f}-{r_hi:.3f}':>13s} "
          f"{(r_hi - r_lo) * LIMB_RADIUS_MM / 0.5:9.2f} mm {sT:9.2f} {sL:9.2f}   "
          f"{'PROVISIONAL' if nm in LIMB.provisional else 'Farina, Mesin, Martina & Merletti (2004)'}")
    r_lo = r_hi

print(f"\nanisotropic layers: {sorted(LIMB.anisotropic_layers)}")
print(f"interface width  epsilon = {LIMB.epsilon:g} of L = {LIMB.epsilon * (N - 1):.2f} h at N = {N}")
print(f"geometry derived from Maksymenko et al. (2023); a {LIMB_RADIUS_MM:.0f} mm limb")

layer              r/R    thickness   sigma_T   sigma_L   source
bone       0.000-0.146      7.00 mm      0.02      0.02   Farina, Mesin, Martina & Merletti (2004)
muscle     0.146-0.417     13.00 mm      0.10      0.50   Farina, Mesin, Martina & Merletti (2004)
fat        0.417-0.479      3.00 mm      0.05      0.05   Farina, Mesin, Martina & Merletti (2004)
skin       0.479-0.500      1.00 mm      1.00      1.00   Farina, Mesin, Martina & Merletti (2004)

anisotropic layers: ['muscle']
interface width  epsilon = 0.00390625 of L = 0.25 h at N = 65
geometry derived from Maksymenko et al. (2023); a 24 mm limb


In [7]:
grid = grid_at(N)
det, rim = grid.det_J, grid.rim_stats(LIMB.shell_norm)

print(grid)
print(f"type(grid) = {type(grid)}")
print(f"rim radius  mean {rim['mean']:.5f}, spread {rim['spread']:.2e} "
      f"({rim['spread_in_h']:.3f} h)")
print(f"det(J)      min {float(det.min()):.5f}, max {float(det.max()):.5f}")
print(f"det(J)      has shape {det.shape}")
print("interfaces  " + ",  ".join(
    f"{a}/{b} {R:.2f} -> {grid.physical_shell_radius(R, LIMB.shell_norm):.4f}"
    for a, b, R in zip(LIMB.layer_names, LIMB.layer_names[1:], LIMB.interfaces)))

CylinderGrid(n=65, L=1.0, smoothing=SmoothingPolicy(iterations=100, del_t=0.2), z_zeta=1.0, mapped=True)
type(grid) = <class 'neuralphysics.grid.CylinderGrid'>
rim radius  mean 0.50144, spread 1.62e-02 (1.035 h)
det(J)      min 0.11417, max 1.11886
det(J)      has shape torch.Size([65, 65, 65])
interfaces  bone/muscle 0.15 -> 0.1463,  muscle/fat 0.42 -> 0.4179,  fat/skin 0.48 -> 0.4805


In [8]:
WIDTH = 0.05      # the 1/e blob radius, a physical length

sources, centres = [], []
for r_norm, angle in SOURCE_SPECS:
    centres.append(dipole_centres(grid, LIMB, r_blob_norm=r_norm, angle_deg=angle))
    sources.append(gaussian_dipole(grid, LIMB, width=WIDTH,
                                   r_blob_norm=r_norm, angle_deg=angle))

W = trapezoid_weight_field(grid.n)
for s, (I, c) in enumerate(zip(sources, centres)):
    clearance = min(c["r_blob"] - c["r_inner"], c["r_outer"] - c["r_blob"])
    print(f"source {s}: r = {c['r_blob']:.4f} at {c['angle_deg']:.0f} deg, "
          f"clearance {clearance:.4f} vs 2w = {2 * WIDTH:.2f}, "
          f"peak {float(I.abs().max()):.4f}, W.I = {float((W * I).sum()):.2e}")

print(f"muscle spans [{centres[0]['r_inner']:.4f}, {centres[0]['r_outer']:.4f}], "
      f"z: source {centres[0]['cz_source']:.3f}, sink {centres[0]['cz_sink']:.3f}, "
      f"w/h = {WIDTH / grid.h:.2f}")

source 0: r = 0.2507 at 0 deg, clearance 0.1045 vs 2w = 0.10, peak 0.9893, W.I = -4.47e-16
source 1: r = 0.3129 at 180 deg, clearance 0.1050 vs 2w = 0.10, peak 0.9871, W.I = -1.29e-17
muscle spans [0.1463, 0.4179], z: source 0.370, sink 0.630, w/h = 3.20


In [9]:
coeff_true = CoefficientHierarchy.from_vectors(LIMB, N)

A = lambda f: laplacian_metric_3D(f, grid.h, coeff_true.faces[grid.n],
                                  coeff_true.metric[grid.n], coeff_true.sigma_T[grid.n])

phi_true = forward(sources[0], coeff_true, num_cycles=NUM_CYCLES)

In [10]:
array = ElectrodeArray()          # full coverage of the four lateral faces
mask_nodes = array.mask(grid.n)

observations = []
for s, I in enumerate(sources):
    phi_s = phi_true if s == 0 else forward(I, coeff_true, num_cycles=NUM_CYCLES)
    phi_obs, noise_stats = add_measurement_noise(phi_s, NOISE_PERCENT, NOISE_SEED + s)
    observations.append(Observation(phi=phi_obs, mask=mask_nodes, grid=grid))
    print(f"noise  source {s}: {NOISE_PERCENT:.1f}% of RMS {noise_stats['rms']:.3e} -> "
          f"std {noise_stats['noise_std']:.3e}, realised SNR {noise_stats['snr']:.2f}")

obs = observations[0]      # the field and residual figures are drawn for source 0
print(f"array  step {array.step}, coverage {array.coverage(grid.n):.3f}, "
      f"{int(obs.mask.sum())} electrodes of {grid.n ** 3} nodes")

noise  source 0: 5.0% of RMS 5.161e-04 -> std 2.581e-05, realised SNR 20.00
noise  source 1: 5.0% of RMS 4.928e-04 -> std 2.464e-05, realised SNR 20.02
array  step 1, coverage 1.000, 16640 electrodes of 274625 nodes


In [ ]:
# the starting point, redrawn from averaged-runs.ipynb's seed. sample_initial_parameters
# draws the runs in sequence from one generator, so drawing N_RUNS of them and taking
# the RUN-th is the same vector that notebook's run 1 started from -- which is why
# N_RUNS is carried here at all, for a notebook that makes a single run.

starts = sample_initial_parameters(LIMB, N_RUNS, seed=ENSEMBLE_SEED,
                                   log_spread=LOG_SPREAD,
                                   radius_spread=RADIUS_SPREAD_MM / MM_PER_UNIT)
theta0 = starts[RUN - 1]

# the labels and the references, taken off the tissue model rather than restated, so
# the table below cannot disagree with the one averaged-runs.ipynb prints
param_names = parameter_names(LIMB)
sigma_true = true_sigma(LIMB)
thick_true = thicknesses(LIMB.interfaces, scale=MM_PER_UNIT)

n_sigma = len(param_names)
sigma_0 = theta0[:n_sigma].exp()
_, _, R_0 = unpack(theta0, LIMB)
thick_0 = thicknesses(R_0, scale=MM_PER_UNIT)

print(f"recovering {n_sigma} conductivities and 2 interface radii, "
      f"{len(theta0)} parameters in all")
print(f"starting from run {RUN} of {N_RUNS} drawn at seed {ENSEMBLE_SEED}\n")
print(f"{'':8s}" + "".join(f"{nm:>10s}" for nm in param_names)
      + "   " + "".join(f"{nm:>9s}" for nm in LIMB.layer_names[:-1]))
print(f"{'':8s}" + "".join(f"{'(S/m)':>10s}" for _ in param_names)
      + "   " + "".join(f"{'(mm)':>9s}" for _ in LIMB.layer_names[:-1]))
print(f"{'start':8s}" + "".join(f"{float(v):10.4f}" for v in sigma_0)
      + "   " + "".join(f"{float(v):9.2f}" for v in thick_0[:-1]))
print(f"{'true':8s}" + "".join(f"{float(v):10.4f}" for v in sigma_true)
      + "   " + "".join(f"{float(v):9.2f}" for v in thick_true[:-1]))

recovering 5 conductivities and 2 interface radii, 7 parameters in all
starting from run 1 of 5 drawn at seed 12345

              bone  muscle_T  muscle_L       fat      skin        bone   muscle      fat
             (S/m)     (S/m)     (S/m)     (S/m)     (S/m)        (mm)     (mm)     (mm)
start       0.0110    0.0669    0.9610    0.0736    0.7872        6.50    13.80     2.71
true        0.0200    0.1000    0.5000    0.0500    1.0000        7.00    13.00     3.00


In [12]:
mask = obs.mask
phi_obs_norm = float(torch.sqrt(sum((o.phi[mask] ** 2).sum() for o in observations)))

print(f"{int(mask.sum())} electrode nodes | ||phi_obs|| on the mask = {phi_obs_norm:.4e}")

16640 electrode nodes | ||phi_obs|| on the mask = 4.9419e-02


## Tikhonov regularisation

The objective above is a pure data misfit. Adding a penalty on the distance from a
reference conductivity, $J_\lambda(\theta) = J(\theta) + \tfrac{1}{2}\lambda\|\theta - \theta_{\rm ref}\|^2$,
trades fit against solution norm.

In [13]:
# one inversion, with the penalty on. Everything else -- the observations, the
# solver, the budget, the tolerances, the starting point -- is what averaged-runs.ipynb
# gave run 1, so the difference between this table and that notebook's is the penalty
# and nothing else.
#
# rho_ref is left at its default, which is theta0's log-conductivities: the penalty
# pulls back towards the start, which is the reading of Tikhonov this experiment means.

run = invert(theta0, LIMB, N, sources, observations,
             num_cycles=NUM_CYCLES, max_iters=MAX_ITERS,
             ftol=FTOL, gtol=GTOL, tikhonov=TIKHONOV)

# the exit report, so "it converged" is checkable rather than assumed. max|g| is
# printed rather than only tested against GTOL: the gradient here is inexact -- the
# frozen diagonal in probe_diagonal, and a forward solve on a fixed cycle budget -- so
# it has a floor under it that no amount of restarting gets below, and where that
# floor sits is worth reading. Note the objective includes the penalty, so the final
# value is not comparable to an unregularised data misfit.
print(f"{run.n_iterations} iterations, {run.n_evaluations} evaluations, "
      f"{run.n_restarts} restarts")
print(f"max|g| = {run.grad_max:.2e} against gtol {GTOL:g}")
print(f"exit: {run.message}")

53 iterations, 158 evaluations, 4 restarts
max|g| = 2.87e-09 against gtol 1e-12
exit: ABNORMAL: 


In [14]:
# the result: the recovered parameters against the truth. One run, so there is no
# spread to quote -- the relative error is this run's, not a mean over runs, and the
# comparison to make is against the per-run rows of averaged-runs.ipynb rather than
# against its ensemble mean.

# a conductivity is quoted to four decimals and a thickness to two: a tenth of a
# micron of muscle is not a claim this experiment is making
sigma_hat = run.sigma
thick_hat = thicknesses(run.interfaces, scale=MM_PER_UNIT)

values = [(f"{nm} (S/m)", rec, ref, ".4f")
          for nm, rec, ref in zip(param_names, sigma_hat, sigma_true)]
values += [(f"{nm} (mm)", rec, ref, ".2f")
           for nm, rec, ref in zip(LIMB.layer_names[:-1], thick_hat, thick_true)]

# formatted once, so the printed table and the saved figure cannot disagree
HEADERS = ("quantity", "recovered", "true", "rel err")
RULE_AFTER = (len(param_names) - 1,)   # conductivities above, thicknesses below
CAPTION = (f"one run, Tikhonov lambda = {TIKHONOV:.0e}, {NOISE_PERCENT:.1f}% noise\n"
           f"started from run {RUN} of the ensemble in averaged-runs.ipynb")
table = [(name, format(float(rec), fmt), format(float(ref), fmt),
          f"{abs(float(rec) - float(ref)) / float(ref):.2%}")
         for name, rec, ref, fmt in values]

widths = [max(len(h), *(len(row[j]) for row in table)) for j, h in enumerate(HEADERS)]
def as_line(cells):
    return "  ".join(c.ljust(w) if j == 0 else c.rjust(w)
                     for j, (c, w) in enumerate(zip(cells, widths)))

print(CAPTION + "\n")
print(as_line(HEADERS))
print("-" * len(as_line(HEADERS)))
for row in table:
    print(as_line(row))

fig_table = viz.plot_table(HEADERS, table, title=CAPTION, rule_after=RULE_AFTER)
fig_table.savefig(RESULTS / "tikhonov-parameter-recovery.png", dpi=200,
                  bbox_inches="tight")
plt.close(fig_table)   # the printed table above is this notebook's copy of it

print(f"\nsaved to {(RESULTS / 'tikhonov-parameter-recovery.png').resolve()}")

one run, Tikhonov lambda = 1e-06, 5.0% noise
started from run 1 of the ensemble in averaged-runs.ipynb

quantity        recovered    true  rel err
------------------------------------------
bone (S/m)         0.0115  0.0200   42.72%
muscle_T (S/m)     0.0968  0.1000    3.21%
muscle_L (S/m)     0.5083  0.5000    1.67%
fat (S/m)          0.0593  0.0500   18.54%
skin (S/m)         1.0897  1.0000    8.97%
bone (mm)            7.41    7.00    5.90%
muscle (mm)         12.30   13.00    5.36%
fat (mm)             3.28    3.00    9.48%

saved to ./experiment-results/tikhonov-parameter-recovery.png
